# 🚢 Lab 2 — The Full Pipeline: Titanic, from Raw Data to Model
In Lab 1 you learned the tools. Today you learn **the recipe** — the sequence of steps that EVERY data project follows:

> **1. Frame the problem → 2. Load & first look → 3. Clean → 4. Explore (EDA) → 5. Prepare features → 6. Split → 7. Train → 8. Evaluate → 9. Interpret → 10. Conclude**

This notebook is a **template you can reuse for future use cases, and those steps are a must in any data science project.** I wrote the structure and most of the code; you complete the numbered **`👉 TODO`** cells.

⚠️ **Complete the TODOs in order** — later cells depend on earlier ones, so cells will error until the TODOs before them are done.

**How you'll be assessed:** the TODOs, the written insight/answer cells (✍️), and the business questions in STEP 11. Code that runs but that you can't explain out loud scores zero.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()
np.random.seed(42)   # reproducibility — same results every run

## STEP 1 — Frame the problem 🎯
Before touching data, answer: *what are we trying to find out, and what does the answer look like?*
- **Analytics question:** which kinds of passengers survived? (insights for humans)
- **Prediction question:** given a passenger's info, predict `Survived` (0 = no, 1 = yes) → **binary classification**, our target column is `Survived`.

| column | meaning |
|---|---|
| Survived | 0 = died, 1 = survived (🎯 target) |
| Pclass | ticket class: 1 = first (rich) … 3 = third |
| Sex, Age | demographics |
| SibSp / Parch | # siblings+spouse / # parents+children aboard |
| Fare | ticket price |
| Embarked | port: C = Cherbourg, Q = Queenstown, S = Southampton |
| PassengerId / Name / Ticket / Cabin | identifiers & extras |

In [ ]:
# 👉 TODO 0 — Write the problem statement in ONE sentence, as a comment:
# "We predict ______ for ______, using ______, so that ______."
...

## STEP 2 — Load & first look 🔍
Always the same ritual from Lab 1: `head`, `shape`, `info`, `describe`.

In [ ]:
try:
    df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
except Exception:
    df = pd.read_csv("https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/data/titanic.csv")   # backup mirror, same file
print("shape:", df.shape)
df.head()

In [ ]:
df.info()   # 👀 look closely: which columns have LESS than 891 non-null values?

In [ ]:
df.describe()

In [ ]:
# 👉 TODO 1 — What fraction of passengers survived?
# Hint: Survived is 0/1, so its .mean() IS the survival rate.
survival_rate = ...
survival_rate

In [ ]:
# 👉 TODO 1b — Write down, as a comment, the accuracy of a model that predicts
# "died" for EVERY passenger. (You will compare your real model against this in STEP 8.)
...

## STEP 3 — Cleaning 🧹
Cleaning = making the data *trustworthy*. The checklist: **missing values → useless columns → duplicates → wrong types.**

In [ ]:
missing = df.isna().sum()
pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing": missing, "percent": pct})[missing > 0]

### The decision framework 📌 (memorize this table)
| Situation | Action | Here |
|---|---|---|
| Column mostly empty (> ~70%) | **drop the column** | Cabin (77% missing) |
| Numeric column, some holes | **fill with median** (robust to outliers — Lab 1!) | Age |
| Category column, few holes | **fill with mode** (most frequent) | Embarked |
| Why not just drop all rows with any NaN? | you'd throw away good data | see below ⬇ |

In [ ]:
print("rows if we dropna:", len(df.dropna()), "/out of", len(df),
      "-> we would lose", len(df) - len(df.dropna()), "passengers. NO.")

In [ ]:
# Drop columns we can't learn from:
#  - Cabin: 77% missing
#  - PassengerId / Ticket: unique IDs — an ID is different for every row, so it can't generalize
#  - Name: text, unique per person (feature engineering on titles is possible — later in your career!)
df = df.drop(columns=["Cabin", "PassengerId", "Ticket", "Name"])
df.head(3)

In [ ]:
# 👉 TODO 2 — Fill missing Age with the MEDIAN age (Lab 1, Exercise 7 pattern)
...

In [ ]:
# 👉 TODO 3 — Fill missing Embarked with the MODE
# Hint: df["Embarked"].mode()[0]
...

In [ ]:
# 👉 TODO 4 — Prove the table is now complete (no missing values anywhere)
...

In [ ]:
# 👉 TODO 4b — In comments, answer:
# a) Why MEDIAN for Age and not MEAN? (use the Lab 1 CEO demo)
# b) Why can't we use a median for Embarked at all?
# c) What did we lose by filling ~20% of Age with one single value?
...

In [ ]:
print("duplicate rows:", df.duplicated().sum())
# a few identical rows can legitimately exist here (same class/age/fare...), so we leave them

## STEP 4 — EDA: interrogate the data 🕵️
EDA has two halves: **(a)** look at each column alone, **(b)** look at everything *against the target*. The single most useful pattern in analytics:

```python
df.groupby(<category>)["Survived"].mean()   # survival RATE per group
```

In [ ]:
sns.countplot(data=df, x="Survived")
plt.title("Class balance: how many died vs survived")
plt.show()

In [ ]:
df[["Age", "Fare", "SibSp", "Parch"]].boxplot()
plt.title("Spread and outliers of the numeric columns")
plt.show()

In [ ]:
# THE golden pattern — survival rate by group:
print(df.groupby("Sex")["Survived"].mean().round(3))
# women vs men — "Women and children first" was real.

In [ ]:
# 👉 TODO 5 — Survival rate by Pclass (the groupby pattern) + a bar chart of it.
# Hint for the chart:  <your groupby result>.plot(kind="bar")
...

In [ ]:
# 👉 TODO 6 — Survival rate by Embarked port. Any difference?
...

In [ ]:
# 👉 TODO 6b — Survival rate by Pclass AND Sex together (group by two columns).
# Which combination had the worst chances? Write it as a comment.
...

In [ ]:
sns.histplot(data=df, x="Age", hue="Survived", bins=30)
plt.title("Age distribution: survivors vs not")
plt.show()
# notice the small bump of survivors at very young ages -> children were prioritized

In [ ]:
# Turning a number into groups with pd.cut — very useful trick (you'll need it again in Lab 3 😉)
df["AgeGroup"] = pd.cut(df["Age"], bins=[0, 12, 18, 35, 60, 80],
                        labels=["child", "teen", "young adult", "adult", "senior"])
df.groupby("AgeGroup", observed=True)["Survived"].mean().round(3)

In [ ]:
# 👉 TODO 7 — Compare the FARE of survivors vs non-survivors.
# a) Compute the mean AND the median fare for each group.
# b) Which of the two should you report, and why? (Lab 1 logic — check the boxplot above.)
...

In [ ]:
# 👉 TODO 7b — Family size: create a column "family_size" = SibSp + Parch + 1 (the passenger).
# Then show the survival rate per family_size. Is travelling alone good or bad?
...

### 👉 TODO 8 — Your insights log ✍️
Every EDA must end with written findings. Write at least **4**, each with the number that supports it:

1.
2.
3.
4.

*(Format: "Sex is the strongest factor — women survived at X% vs men at Y%.")*

## STEP 5 — Prepare features for the model 🔧
Models eat **numbers only** (remember: your X in the course was a numeric matrix). Two moves:
1. Drop helper columns we made just for EDA (`AgeGroup`).
2. Convert text categories to 0/1 columns with **`pd.get_dummies`**: `Sex` becomes `Sex_male` (1 = male). `drop_first=True` avoids a redundant column (if not male → female, no need for both).

In [ ]:
X = pd.get_dummies(
    df.drop(columns=["Survived", "AgeGroup"]),
    drop_first=True
).astype(float)

y = df["Survived"]

In [ ]:
# 👉 TODO 9 — Sanity check: print X.shape and X.dtypes. Everything numeric?
...

In [ ]:
# 👉 TODO 9b — In comments:
# a) What would LogisticRegression do with the raw text value "male"? Why does it need dummies?
# b) Run get_dummies again with drop_first=False and print the columns.
#    Name one pair of columns that is redundant, and explain the redundancy in one line.
...

## STEP 6 — Train / test split ✂️
You know this from Course 2: **never grade a model on data it trained on.** We hide 20% as a test set. `stratify=y` keeps the survived/died ratio identical in both parts; `random_state=42` makes the split reproducible.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("train:", X_train.shape, " test:", X_test.shape)

In [ ]:
# 👉 TODO 10 — Prove stratify worked: print the survival rate in y_train and in y_test.
# Then re-split ONCE with stratify=None and a different random_state, and print them again.
# How much did the test survival rate move? Comment on why that would be a problem.
...

### ⚠️ The leakage note (important — this comes back in the leakage lecture)
Above, we filled Age with the median of the **whole table**, before splitting. Strictly, that is a small **data leak**: the test rows contributed to a number used in preprocessing, and in production you could never compute a statistic using future/unseen data.

The production-grade order is: **split first → fit statistics on TRAIN only → apply them to test.**

In [ ]:
# 👉 TODO 10b — Write the leak-free version as code (you can run it on a fresh copy):
#   age_median = X_train["Age"].median()      # fitted on TRAIN only
#   X_train["Age"] = X_train["Age"].fillna(age_median)
#   X_test["Age"]  = X_test["Age"].fillna(age_median)   # applied, not re-fitted
# Then answer in a comment: the "golden question" for any feature is
# "at prediction time, will I truly have this value, computed exactly this way?" — 
# explain why the whole-table median fails that test.
...

## STEP 7 — Train the model 🤖
`LogisticRegression` — literally the algorithm from your course: sigmoid, log loss, gradient-descent-style optimization, all wrapped in 2 lines.

📌 *Course connection:* sklearn applies **L2 regularization by default** (parameter `C` — careful, it's the *inverse* of λ: big C = weak regularization).

In [ ]:
# 👉 TODO 11 — Create and train the model:
#   model = LogisticRegression(max_iter=1000)
#   then .fit(...) it on the TRAINING data
from sklearn.linear_model import LogisticRegression
...

In [ ]:
# 👉 TODO 12 — Predict on the TEST set, store as y_pred
...

In [ ]:
# 👉 TODO 12b (stretch) — Try three values of C: 0.01, 1, 100.
# Print the training and test accuracy for each.
# Which direction is more regularization, and what happens to the gap between train and test?
...

## STEP 8 — Evaluate honestly 📏
First rule of evaluation: **beat the dumb baseline.** A "model" that predicts *everyone died* is right ~61.6% of the time here. If we can't clearly beat that, we've built nothing.

In [ ]:
baseline = max(y_test.mean(), 1 - y_test.mean())
print(f"dumb baseline accuracy: {baseline:.1%}")

In [ ]:
# 👉 TODO 13 — Compute the model's accuracy on the test set, and print the
# LIFT over the baseline in percentage points.
from sklearn.metrics import accuracy_score
...

In [ ]:
# The confusion matrix — WHERE does the model make mistakes?
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["died", "survived"])
plt.title("rows = truth, columns = prediction")
plt.show()

In [ ]:
# Precision & recall — the Course 2 metrics, computed per class:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, target_names=["died", "survived"]))
# precision (survived) = of those we PREDICTED survived, how many really did
# recall    (survived) = of those who REALLY survived, how many we found

In [ ]:
# 👉 TODO 14 — Read your confusion matrix and write, as comments:
# a) how many real survivors the model MISSED (false negatives)
# b) how many people it wrongly labelled as survivors (false positives)
# c) compute precision and recall for "survived" BY HAND from the four cells,
#    and check they match the report above.
...

In [ ]:
# 👉 TODO 14b — Threshold play:
# Use model.predict_proba(X_test)[:, 1] and classify with a threshold of 0.3 instead of 0.5.
# Print the new classification report. What happened to recall and precision, and why?
...

### ✍️ Think & answer (in this cell):
Imagine the model decides who gets a rescue-team visit first. Missing a real survivor is catastrophic; visiting someone who didn't need it just wastes time. Would you rather maximize **precision** or **recall** for the "survived" class? Why?

*Your answer:*

## STEP 9 — Interpret the model 🔎
Logistic regression's superpower: its weights **w** (course notation!) are readable. Positive weight → pushes toward survival; negative → toward death.

In [ ]:
weights = pd.Series(model.coef_[0], index=X.columns).sort_values()
print(weights.round(3))
weights.plot(kind="barh")
plt.title("Model weights (w)")
plt.show()

In [ ]:
# 👉 TODO 15 — Do the weights agree with your EDA findings from TODO 8?
# Name one feature where they agree and, if any, one that surprised you. (comments)
...

## STEP 10 — Conclude 📝 ✍️
Every notebook ends with a summary a non-technical person could read. Fill this in (4–6 sentences): *what was the question, what did the data show, how good is the model, what would you do next?*

**My conclusion:**

*(write here)*

## STEP 11 — 🏆 Business questions
Now step out of the notebook and into the meeting room. Answer each in the markdown cell below it, in plain English — numbers allowed, jargon not. If the answer is "the data can't tell us", say that and say what you'd need.

In [ ]:
# ✏️ Business Q1: A modern cruise line asks you to build this model for THEIR ships,
#    to decide lifeboat priority. Compute anything you need, then answer below:
#    would you deploy it? What is the strongest argument against doing so?
...

✍️ **Your answer (Q1):**

In [ ]:
# ✏️ Business Q2: Your model uses Sex as a feature and it is the strongest predictor.
#    a) Is using Sex acceptable HERE (a historical analysis)?
#    b) Would it be acceptable in a model that decides who gets a bank loan? Why the difference?
#    (Think: is this a relevance decision or an opportunity decision?)
...

✍️ **Your answer (Q2):**

In [ ]:
# ✏️ Business Q3: The captain's office wants ONE number to report: "our model is X% accurate."
#    a) Give the number.
#    b) Explain in 2 sentences why that single number is the least useful thing you computed today,
#       using the baseline and the confusion matrix as evidence.
...

✍️ **Your answer (Q3):**

In [ ]:
# ✏️ Business Q4: Someone proposes adding a feature called `rescued_by_boat_number`
#    (filled only for passengers who made it onto a lifeboat). Model accuracy jumps to 99%.
#    Explain why this feature must be rejected, using the golden question from TODO 10b.
...

✍️ **Your answer (Q4):**

In [ ]:
# ✏️ Business Q5: The fare column is heavily skewed (check the boxplot in STEP 4).
#    a) Create a log-transformed fare: np.log1p(df["Fare"]) and compare the two distributions.
#    b) Retrain with the log version instead. Did anything change?
#    c) In one sentence: why might a skewed feature bother a linear model?
...

✍️ **Your answer (Q5):**

In [ ]:
# ✏️ Business Q6 (the scoping question): Management asks for "a model to predict who will
#    survive a future disaster on a modern ship."
#    List THREE reasons this 1912 dataset may not generalize to that problem.
#    (Think about: the population, the technology, the rules, the time gap.)
...

✍️ **Your answer (Q6):**

## 📋 The template you now own
1. **Frame** — what are we predicting, for whom, and what decision does it drive?
2. **Load & first look** — head / shape / info / describe
3. **Clean** — missing → useless columns → duplicates → types (and *document every decision*)
4. **EDA vs target** — `groupby(category)[target].mean()` + charts that changed your mind
5. **Feature prep** — dummies, drop helpers, everything numeric
6. **Split** — stratified, reproducible, *before* fitting any statistic
7. **Train** — start simple (logistic regression is a feature, not a weakness)
8. **Evaluate** — versus a baseline, with the confusion matrix, precision and recall
9. **Interpret** — do the weights agree with the EDA?
10. **Conclude** — 5 sentences a non-technical manager can act on

Every project you build from now on is this skeleton with different data.